# Proyecto final integrador · Fase 4

## Factores sociodemográficos y su asociación con factores de riesgo cardiovascular, utilizando la Encuesta Nacional de Salud (ENS) 2016-2017 de Chile.

**MCDI500 · Programación para la Ciencia de Datos · Grupo 4**

Integrantes: Felipe Díaz Toro, Ninoska Yévenes Hernández, Guillermo Subiabre Chacón, Pablo Ríos Passteni
Docente: Omar Salinas

---
## Parte 0 · Entorno y reproducibilidad

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")

print("REGISTRO DEL ENTORNO")
print("=" * 58)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  NumPy       : {np.__version__}")
print(f"  pandas      : {pd.__version__}")
print(f"  Matplotlib  : {plt.matplotlib.__version__}")
print(f"  seaborn     : {sns.__version__}")
print(f"  Entorno virtual activo: {sys.prefix != sys.base_prefix}")

---
# I. Fase 1 · El problema y los objetivos

## 1.1 Contexto y problema

Las enfermedades cardiovasculares son la principal causa de muerte en Chile y en el mundo.
Según la Encuesta Nacional de Salud (ENS) 2016-2017 del Ministerio de Salud, la prevalencia
nacional de hipertensión arterial alcanza 27,6% y la de sospecha de diabetes mellitus 12,3%
en personas de 15 años y más (MINSAL, 2017). Estas cifras agregadas no informan cómo se
distribuye el riesgo al interior de la población.

**Problema.** Se dispone de registros individuales de la ENS 2016-2017, pero no de una
caracterización de cómo el perfil sociodemográfico de las personas encuestadas se asocia
con los principales indicadores de riesgo cardiovascular disponibles en la encuesta.

## 1.2 Pregunta de investigación

> ¿Cómo se relacionan las condiciones sociodemográficas (edad, sexo, nivel educacional,
> ingreso del hogar y zona de residencia) con la presencia de hipertensión arterial,
> diabetes, colesterol alto, índice de masa corporal y nivel de actividad física en los
> participantes de la ENS 2016-2017?

## 1.3 Objetivo general

> Caracterizar la relación entre las condiciones sociodemográficas de los participantes
> (edad, sexo, nivel educacional, ingreso del hogar y zona de residencia) y la presencia de
> hipertensión arterial, diabetes, colesterol alto, un índice de masa corporal elevado y un
> nivel insuficiente de actividad física en la ENS 2016-2017

## 1.4 Objetivos específicos

1. Comparar la prevalencia de hipertensión arterial entre los distintos grupos etarios.
2. Describir cómo varía la prevalencia de diabetes según el grupo etario y la zona de residencia.
3. Analizar la relación entre el nivel educacional y la presencia de colesterol alto.
4. Examinar la asociación entre el índice de masa corporal y el nivel de actividad física.
5. Evaluar si la relación entre edad e índice de masa corporal, en presencia de hipertensión, se mantiene al estratificar por sexo.

---
# II. Fase 2 · El conjunto preparado

La Fase 2 no se repite acá: se carga su resultado y se documentan las decisiones que lo
produjeron.

## 2.1 Decisiones de preparación y su justificación

| Decisión | Alternativa descartada | Razón |
|---|---|---|
| Conservar los registros sin ingreso (`as27`/`as28`), marcados con bandera | Eliminar las filas | Los faltantes no son aleatorios |
| Imputar `as27` por mediana dentro de su propio tramo (`as28`) | Imputar con la mediana global | Preserva la estructura por tramo de ingreso |
| Codificar Sexo, Zona, diabetes y colesterol con one-hot | Codificación ordinal | Son categorías nominales, sin orden |
| Escalar Edad, IMC e ingreso con media y desviación | Dejar las unidades originales | Requerido por Fase 3 |
| Reclasificar GPAQ de nominal a ordinal | Tratarla como nominal | Bajo < Moderado < Alto tiene orden |

**Consecuencia para la Fase 4.** Las dos últimas decisiones dejan el conjunto ilegible para
graficar (edad e IMC centrados en cero, sexo convertido en columnas de ceros y unos). La
Parte IV resuelve esto.

## 2.2 Carga del conjunto de la Fase 2

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent.parent  # ajusta si tu notebook no está en F4/notebooks/
sys.path.insert(0, str(RAIZ))

from src.carga import leer_archivo

datos = leer_archivo(str(RAIZ / "data" / "processed" / "ens_variables_f1f2.xlsx"))
print(f"Conjunto de origen (F1-F2): {datos.shape[0]} filas x {datos.shape[1]} columnas")
datos.head(3)

---
# III. Fase 3 · El núcleo algorítmico

Se integran las clases de `src/` que ya construyeron y se verifica que el pipeline
reproduce exactamente el resultado oficial de la Fase 2.

## 3.1 El pipeline completo

In [ ]:
from src.imputadores import MarcadorNoRespuesta, EliminadorFilasNulas, ImputadorFlexible
from src.imputadores import PorMediana, PorMedianaDeTramo, PorModa
from src.transformadores import CodificadorOneHot, EscaladorEstandar, EliminadorColumna, ConvertidorEntero
from src.pipeline import Pipeline

variables_categoricas = ["Sexo", "Zona", "di3", "dis2"]
variables_numericas = ["Edad", "IMC", "as27"]

pasos_limpieza = [
    MarcadorNoRespuesta("as28"),
    EliminadorFilasNulas("HTA"),
    ImputadorFlexible("IMC", PorMediana()),
    ImputadorFlexible("anos_estudio_MINSAL_1", PorMediana()),
    ImputadorFlexible("as27", PorMedianaDeTramo("as28")),
    ImputadorFlexible("GPAQ", PorModa()),
]

pasos_integracion = (
    [EliminadorColumna(col) for col in ["IdEncuesta", "FechaInicioF1"]]
    + [ConvertidorEntero(col) for col in ["HTA", "GPAQ"]]
    + [CodificadorOneHot(col) for col in variables_categoricas]
    + [EscaladorEstandar(col) for col in variables_numericas]
)

pipeline_completo = Pipeline(pasos_limpieza + pasos_integracion)
df_transformado = pipeline_completo.ajustar_transformar(datos)

print(f"Dimensiones iniciales : {datos.shape}")
print(f"Dimensiones finales   : {df_transformado.shape}")

### 3.2 Eficiencia computacional

Además de validar el pipeline, se comprueba brevemente el costo computacional de los transformadores propios frente a una implementación de referencia de scikit-learn/pandas. El análisis completo (variación de tamaño de muestra, medición de memoria con `tracemalloc`, y evaluación de complejidad) se realizó en Fase 3 y está documentado en `F3/notebooks/S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb`. Aquí se replica la comparación de tiempos sobre el conjunto real de este proyecto (5.511 filas), como evidencia directa dentro del notebook consolidado.

In [ ]:
from src.medicion import medir_tiempo_timeit
from sklearn.preprocessing import StandardScaler

def escalar_propio(df, columna):
    return EscaladorEstandar(columna).ajustar_transformar(df[[columna]].copy())

def escalar_sklearn(df, columna):
    resultado = df[[columna]].copy()
    resultado[columna] = StandardScaler().fit_transform(resultado[[columna]])
    return resultado

tiempo_propio = medir_tiempo_timeit(escalar_propio, df_transformado, "Edad")
tiempo_sklearn = medir_tiempo_timeit(escalar_sklearn, df_transformado, "Edad")

pd.DataFrame([
    {"implementacion": "EscaladorEstandar (propio)", "tiempo_s": tiempo_propio},
    {"implementacion": "StandardScaler (sklearn)", "tiempo_s": tiempo_sklearn},
])

## 3.2 Verificación contra el resultado oficial de la Fase 2

In [ ]:
from src.validador import verificar_contra_fase2

RUTA_PROCESADO = RAIZ / "data" / "processed" / "ens_procesado.csv"

pipeline_verificacion = Pipeline(pasos_limpieza + pasos_integracion)
coincide = verificar_contra_fase2(
    pipeline_verificacion,
    datos,
    RUTA_PROCESADO,
    verificar_tipos=True,
)
assert coincide, "El pipeline no reproduce el resultado oficial de la Fase 2"
print("Verificado: el pipeline reproduce exactamente ens_procesado.csv")

## 3.3 Validación técnica (59 pruebas)

Casos normal, límite y excepción, definidos en `tests/casos_por_clase.py`, ejecutados
directamente acá.

In [ ]:
from tests.casos_por_clase import ejecutar_pruebas

reporte = ejecutar_pruebas()
print(reporte.to_string(index=False))

---
# IV. Fase 4 · Conjunto interpretable para visualizar

`df_transformado` tiene la edad e IMC centrados en cero y las categóricas en one-hot —
correcto para el análisis, ilegible para graficar. La solución: capturar el dato en el
punto del pipeline donde ya está limpio pero todavía no se codifica ni escala, justo antes
de `pasos_integracion`. Se reutilizan las mismas clases, sin duplicar lógica de limpieza.

## 4.1 df_visual

In [ ]:
pipeline_limpieza = Pipeline(pasos_limpieza)
df_visual = pipeline_limpieza.ajustar_transformar(datos)

print(f"df_visual: {df_visual.shape[0]} filas x {df_visual.shape[1]} columnas")
print(f"  Edad en anios : {df_visual['Edad'].min():.0f} a {df_visual['Edad'].max():.0f}")
print(f"  IMC           : {df_visual['IMC'].min():.1f} a {df_visual['IMC'].max():.1f}")

## 4.2 Etiquetas legibles según el libro de códigos ENS

Los códigos numéricos de Sexo, Zona, diabetes (`di3`) y colesterol alto (`dis2`) se
traducen a texto, según `docs/diccionario_variables.csv` y el libro de códigos F1 de la
ENS 2016-2017.

In [ ]:
import numpy as np

df_visual["Sexo_txt"] = df_visual["Sexo"].map({1: "Hombre", 2: "Mujer"})
df_visual["Zona_txt"] = df_visual["Zona"].map({1: "Urbano", 2: "Rural"})
df_visual["diabetes_txt"] = df_visual["di3"].map({1: "Si", 2: "No", 3: "No recuerda"})
df_visual["colesterol_txt"] = df_visual["dis2"].map({
    1: "Si, una vez", 2: "Si, mas de una vez", 3: "Nunca", 4: "No recuerda",
})
df_visual["actividad_txt"] = df_visual["GPAQ"].map({1: "Bajo", 2: "Moderado", 3: "Alto"})
df_visual["HTA_txt"] = np.where(df_visual["HTA"] == 1, "Con HTA", "Sin HTA")

TRAMOS = [0, 30, 45, 60, 75, 120]
ETIQUETAS_EDAD = ["Menos de 30", "30 a 44", "45 a 59", "60 a 74", "75 o mas"]
df_visual["grupo_etario"] = pd.cut(df_visual["Edad"], bins=TRAMOS, labels=ETIQUETAS_EDAD, right=False)
df_visual["grupo_etario"] = df_visual["grupo_etario"].cat.set_categories(ETIQUETAS_EDAD, ordered=True)

print("df_visual listo:", df_visual.shape[1], "columnas")
df_visual[["Edad", "grupo_etario", "Sexo_txt", "Zona_txt", "HTA_txt", "diabetes_txt", "colesterol_txt", "actividad_txt"]].head(3)

## 4.3 Ponderación por factor de expansión muestral

La ENS 2016-2017 tiene diseño muestral complejo, así que una tasa calculada directamente
sobre los 5.511 encuestados describe a la muestra, no a la población. Para que las cifras
de este informe sean comparables con las que reporta MINSAL (por ejemplo, el 27,6% de HTA
citado en la introducción), las tasas se ponderan con `Fexp_F1F2p_Corr`.

**Lo que sí y lo que no incluye esta ponderación.** Se pondera el punto estimado (la tasa),
que es lo que exige la comparabilidad con las cifras oficiales. No se estima el error
estándar ni el intervalo de confianza considerando conglomerados y estratos, eso requiere
métodos de varianza por diseño muestral que exceden el alcance de este curso, y queda
declarado como limitación en la discusión.

In [ ]:
assert "Fexp_F1F2p_Corr" in df_visual.columns, "Falta el ponderador muestral"

def resumir_por(df, variable, objetivo, ponderador="Fexp_F1F2p_Corr"):
    filas = []
    for valor, grupo in df.groupby(variable, observed=True):
        tasa = np.average(grupo[objetivo], weights=grupo[ponderador]) * 100
        filas.append({variable: valor, "personas": len(grupo), "tasa": round(tasa, 2)})
    return pd.DataFrame(filas).set_index(variable)

# Prueba rapida con HTA
print(resumir_por(df_visual, "grupo_etario", "HTA").to_string())

---
# V. Exploración dirigida por los objetivos

Antes de pulir las figuras finales del informe, se recorren los cinco objetivos específicos
con una tabla o un gráfico rápido cada uno, usando `df_visual` y `resumir_por()` ya
construidos en la Parte IV. Esto deja evidencia de que los cinco fueron explorados, aunque
solo tres se conviertan en figura oficial en la Parte VI.

In [ ]:
DESTACADO = "#AC212E"   # aquello sobre lo que se dirige la mirada
NEUTRO    = "#9FB6C8"   # el contexto

# Banderas binarias para diabetes y colesterol alto, necesarias para resumir_por()
# (di3 y dis2 no son 0/1: se simplifican a presencia/ausencia, declarado como limitacion)
df_visual["diabetes"] = (df_visual["di3"] == 1).astype(int)
df_visual["colesterol_alto"] = df_visual["dis2"].isin([1, 2]).astype(int)

print("Banderas creadas: diabetes, colesterol_alto")

## 5.1 Objetivo 1 · Prevalencia de HTA por grupo etario

In [ ]:
resumen_hta_edad = resumir_por(df_visual, "grupo_etario", "HTA")
print(resumen_hta_edad.to_string())

fig, eje = plt.subplots(figsize=(7, 4))
eje.bar(resumen_hta_edad.index.astype(str), resumen_hta_edad["tasa"], color=NEUTRO)
eje.set_title("Objetivo 1: prevalencia de HTA por grupo etario")
eje.set_xlabel("Grupo etario"); eje.set_ylabel("Prevalencia de HTA (%, ponderada)")
plt.tight_layout(); plt.show()

## 5.2 Objetivo 2 · Prevalencia de diabetes por grupo etario y zona

In [ ]:
filas = []
for (grupo, zona), sub in df_visual.groupby(["grupo_etario", "Zona_txt"], observed=True):
    tasa = np.average(sub["diabetes"], weights=sub["Fexp_F1F2p_Corr"]) * 100
    filas.append({"grupo_etario": grupo, "Zona": zona, "tasa": round(tasa, 2), "personas": len(sub)})
resumen_diabetes = pd.DataFrame(filas)
print(resumen_diabetes.to_string(index=False))

fig, eje = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=resumen_diabetes, x="grupo_etario", y="tasa", hue="Zona",
            palette=[NEUTRO, "#5A7A94"], ax=eje)  # dos tonos neutros: no hay un hallazgo puntual que destacar aqui
eje.set_title("Objetivo 2: prevalencia de diabetes por grupo etario y zona")
eje.set_xlabel("Grupo etario"); eje.set_ylabel("Prevalencia de diabetes (%, ponderada)")
eje.legend(title="Zona")
plt.tight_layout(); plt.show()

## 5.3 Objetivo 3 · Nivel educacional según colesterol alto

In [ ]:
orden_colesterol = ["Nunca", "No recuerda", "Si, una vez", "Si, mas de una vez"]

resumen_educacion = (
    df_visual.groupby("colesterol_txt", observed=True)["anos_estudio_MINSAL_1"]
    .agg(personas="size", promedio="mean", mediana="median").round(1)
    .reindex(orden_colesterol)
)
print(resumen_educacion.to_string())

fig, eje = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df_visual, x="colesterol_txt", y="anos_estudio_MINSAL_1",
            order=orden_colesterol, color=NEUTRO, ax=eje)
eje.set_title("Objetivo 3: anios de estudio segun colesterol alto")
eje.set_xlabel("Colesterol alto"); eje.set_ylabel("Anios de estudio")
plt.tight_layout(); plt.show()

## 5.4 Objetivo 4 · IMC según nivel de actividad física

In [ ]:
orden_actividad = ["Bajo", "Moderado", "Alto"]

fig, eje = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df_visual, x="actividad_txt", y="IMC", order=orden_actividad,
            color=NEUTRO, fliersize=0, ax=eje)
sns.stripplot(data=df_visual.sample(min(500, len(df_visual)), random_state=2026),
              x="actividad_txt", y="IMC", order=orden_actividad,
              size=2.5, color="#0A1526", alpha=0.25, jitter=0.25, ax=eje)
eje.set_title("Objetivo 4: IMC segun nivel de actividad fisica")
eje.set_xlabel("Actividad fisica"); eje.set_ylabel("IMC")
plt.tight_layout(); plt.show()

## 5.5 Objetivo 5 · Edad, IMC y HTA estratificado por sexo (posible confusión de Simpson)

In [ ]:
grilla = sns.FacetGrid(df_visual, col="Sexo_txt", hue="HTA_txt",
                        palette=[NEUTRO, DESTACADO], height=4, aspect=1.1)
grilla.map(plt.scatter, "Edad", "IMC", alpha=0.35, s=14)
grilla.add_legend(title="Diagnostico")
grilla.set_axis_labels("Edad (anios)", "IMC")
grilla.set_titles("{col_name}")
plt.tight_layout(); plt.show()

### Selección de las tres figuras oficiales

De los cinco objetivos explorados, se seleccionan tres para pulir como figuras del informe
(Parte VI), siguiendo el hilo narrativo contexto → contraste → resolución:

1. **Objetivo 1** (HTA por grupo etario) — contexto: conecta con la cifra nacional citada
   en la introducción.
2. **Objetivo 4** (IMC según actividad física) — contraste: un segundo factor de riesgo,
   con un tipo de gráfico distinto.
3. **Objetivo 5** (Simpson por sexo) — resolución: examina si el patrón se sostiene al
   estratificar, el análisis más profundo de los cinco.

Los objetivos 2 y 3 (diabetes por zona, educación y colesterol) quedan documentados acá,
con tabla y gráfico propios, pero no se llevan a figura final: se prioriza profundidad de
interpretación en tres relaciones por sobre cobertura superficial de las cinco.

---
# VI. Las tres figuras del informe

De los cinco objetivos explorados en la Parte V, se pulen tres como figuras oficiales,
siguiendo el hilo narrativo contexto → contraste → resolución (sección 9 de la guía de
Fase 4). Cada una lleva título con el hallazgo, ejes con unidad y fuente al pie.

In [ ]:
import os

CARPETA_FIGURAS = RAIZ / "F4" / "figuras"
os.makedirs(CARPETA_FIGURAS, exist_ok=True)

def rotular(fig, eje, titulo, x, y, fuente="ENS 2016-2017, elaboracion propia"):
    """Aplica el rotulado obligatorio: titulo-hallazgo, ejes con unidad y fuente.

    Centralizarlo evita que alguna figura quede sin titulo, sin unidades o sin fuente.
    """
    eje.set_title(titulo)
    eje.set_xlabel(x)
    eje.set_ylabel(y)
    fig.text(0.01, -0.03, f"Fuente: {fuente}", fontsize=8, color="#5A6B7A")
    return eje

def exportar(fig, nombre_archivo):
    """Guarda la figura en F4/figuras/, para insertarla en el informe sin capturas de pantalla."""
    ruta = CARPETA_FIGURAS / nombre_archivo
    fig.savefig(ruta, dpi=200, bbox_inches="tight")
    print(f"Guardada: {ruta.relative_to(RAIZ)}")

## Figura 1 · Objetivo 1 (contexto): prevalencia de HTA por grupo etario

In [ ]:
resumen_hta_edad = resumir_por(df_visual, "grupo_etario", "HTA")

fig, eje = plt.subplots(figsize=(8, 4.8))
colores = [DESTACADO if v == resumen_hta_edad["tasa"].max() else NEUTRO for v in resumen_hta_edad["tasa"]]
barras = eje.bar(resumen_hta_edad.index.astype(str), resumen_hta_edad["tasa"], color=colores, width=0.65)
eje.bar_label(barras, fmt="%.1f%%", padding=3)
rotular(fig, eje,
        "La prevalencia de HTA pasa de 1,9% en menores de 30 anios a 82,6% en mayores de 75",
        "Grupo etario", "Prevalencia de HTA (%, ponderada)")
eje.set_ylim(0, resumen_hta_edad["tasa"].max() * 1.2)
plt.tight_layout()
exportar(fig, "figura_1_hta_por_edad.png")
plt.show()

**Qué muestra.** La prevalencia de HTA crece de forma sostenida con la edad: es marginal
antes de los 30 años (1,9%) y alcanza a más de 8 de cada 10 personas después de los 75
(82,6%), superando el 50% ya en el tramo de 60 a 74 años.

**Qué se infiere.** El envejecimiento es, con distancia, el factor individual más asociado a
la HTA en este conjunto, consistente con la literatura clínica y con la cifra nacional de
27,6% citada en la introducción, que agrega todos los tramos de edad.

**Qué límite tiene.** Es una asociación bivariada: no descarta que parte del efecto se
explique por otras variables que también cambian con la edad, como el IMC (ver Figura 2) o
los años de estudio.

## Figura 2 · Objetivo 4 (contraste): IMC según nivel de actividad física

In [ ]:
orden_actividad = ["Bajo", "Moderado", "Alto"]
medianas_actividad = df_visual.groupby("actividad_txt", observed=True)["IMC"].median().reindex(orden_actividad)

fig, eje = plt.subplots(figsize=(7.5, 4.8))
colores_caja = {"Bajo": DESTACADO, "Moderado": NEUTRO, "Alto": NEUTRO}
sns.boxplot(data=df_visual, x="actividad_txt", y="IMC", order=orden_actividad,
            hue="actividad_txt", palette=colores_caja, legend=False, fliersize=0, ax=eje)
sns.stripplot(data=df_visual.sample(min(500, len(df_visual)), random_state=2026),
              x="actividad_txt", y="IMC", order=orden_actividad,
              size=2.4, color="#0A1526", alpha=0.2, jitter=0.28, ax=eje)
rotular(fig, eje,
        "El IMC baja levemente a mayor actividad fisica: mediana de 28,7 a 27,9",
        "Nivel de actividad fisica (GPAQ)", "IMC (kg/m2)")
plt.tight_layout()
exportar(fig, "figura_2_imc_por_actividad.png")
plt.show()
print(medianas_actividad.round(1).to_string())

**Qué muestra.** La mediana de IMC baja de 28,7 en actividad baja a 28,1 en moderada y
27,9 en alta. La dirección es consistente, pero la diferencia entre el extremo bajo y el
alto es de menos de un punto de IMC.

**Qué se infiere.** Hay una asociación entre mayor actividad física y menor IMC, coherente con
la evidencia clínica, pero de magnitud modesta en este conjunto: la actividad física por sí
sola no separa claramente los grupos.

**Qué límite tiene.** GPAQ es autorreportado (no medido), y la relación observada no permite
establecer causalidad: personas con IMC más alto también podrían reducir su actividad física
a causa de su condición, no solo al revés.

## Figura 3 · Objetivo 5 (resolución): edad, IMC y HTA, estratificado por sexo

In [ ]:
grilla = sns.FacetGrid(df_visual, col="Sexo_txt", hue="HTA_txt",
                        palette=[NEUTRO, DESTACADO], height=4.3, aspect=1.05)
grilla.map(plt.scatter, "Edad", "IMC", alpha=0.35, s=14)
grilla.add_legend(title="Diagnostico")
grilla.set_axis_labels("Edad (anios)", "IMC (kg/m2)")
grilla.set_titles("{col_name}")
grilla.figure.suptitle(
    "La relacion entre edad, IMC e hipertension se mantiene similar en hombres y mujeres",
    y=1.04)
grilla.figure.text(0.01, -0.03, "Fuente: ENS 2016-2017, elaboracion propia", fontsize=8, color="#5A6B7A")
exportar(grilla.figure, "figura_3_edad_imc_hta_por_sexo.png")
plt.show()

for sexo, g in df_visual.groupby("Sexo_txt"):
    tasa = np.average(g["HTA"], weights=g["Fexp_F1F2p_Corr"]) * 100
    etiqueta = "hombres" if sexo == "Hombre" else "mujeres"
    print(f"Prevalencia de HTA en {etiqueta}: {tasa:.1f}% (n={len(g)})")

**Qué muestra.** En ambos paneles, las personas con HTA (destacadas) se concentran hacia
edades mayores e IMC más alto, y el patrón visual es prácticamente el mismo en hombres que
en mujeres. La prevalencia ponderada de HTA también es casi idéntica entre sexos (27,5% en
hombres, 27,7% en mujeres).

**Qué se infiere.** No se observa una variable de confusión fuerte por sexo: la asociación
entre edad, IMC e hipertensión no cambia de dirección ni de magnitud relevante al
estratificar, a diferencia de lo que ocurriría en un caso clásico de paradoja de Simpson.

**Qué límite tiene.** Esto descarta el sexo como confusor evidente en esta relación
particular, pero no descarta que otras variables no examinadas aquí (por ejemplo, zona o
nivel educacional) sí modifiquen el patrón.